In [ ]:
import io
import zipfile
import requests
import frontmatter


def _download_repo_zip(repo_owner, repo_name):
    """Download repository zip using GitHub default branch automatically."""
    url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/zipball"
    headers = {"User-Agent": "aihero-course-notebook"}
    resp = requests.get(url, headers=headers, timeout=60)

    if resp.status_code != 200:
        raise Exception(
            f"Failed to download repository zip for {repo_owner}/{repo_name}: "
            f"HTTP {resp.status_code}"
        )

    return resp


def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.

    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name

    Returns:
        List of dictionaries containing file content and metadata
    """
    resp = _download_repo_zip(repo_owner, repo_name)

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith(".md") or filename_lower.endswith(".mdx")):
            continue

        with zf.open(file_info) as f_in:
            content = f_in.read().decode("utf-8", errors="ignore")

        # Some repos contain invalid YAML frontmatter.
        # Fallback to raw content so ingestion does not fail.
        try:
            post = frontmatter.loads(content)
            data = post.to_dict()
        except Exception as e:
            data = {
                "content": content,
                "frontmatter_error": str(e),
            }

        data["filename"] = filename
        repository_data.append(data)

    zf.close()
    return repository_data

In [ ]:
dtc_langchain = read_repo_data('langchain-ai', 'langchain')
#dtc_microsoft = read_repo_data('microsoft', 'semantic-kernel')
#dtc_openai = read_repo_data('openai', 'openai-cookbook')
print(f"Langchain documents: {len(dtc_langchain)}")
#print(f"microsoft documents: {len(dtc_microsoft)}")
#print(f"openai documents: {len(dtc_openai)}")

In [ ]:
dtc_microsoft = read_repo_data('microsoft', 'semantic-kernel')
print(f"microsoft documents: {len(dtc_microsoft)}")


In [ ]:
dtc_openai = read_repo_data('openai', 'openai-cookbook') 
print(f"openai documents: {len(dtc_openai)}")

In [ ]:
langchain-ai/langchain
microsoft/semantic-kernel
openai/openai-cookbook

In [ ]:
import io
import zipfile
import requests

def count_markdown_files(owner, repo):
    url = f"https://api.github.com/repos/{owner}/{repo}/zipball"
    resp = requests.get(url, headers={"User-Agent": "aihero-course-notebook"}, timeout=60)
    resp.raise_for_status()

    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    files = [
        f.filename for f in zf.infolist()
        if f.filename.lower().endswith((".md", ".mdx"))
    ]
    zf.close()
    return len(files), files[:20]  # sample first 20

for owner, repo in [
    ("langchain-ai", "langchain"),
    ("microsoft", "semantic-kernel"),
    ("openai", "openai-cookbook"),
]:
    n, sample = count_markdown_files(owner, repo)
    print(f"{owner}/{repo}: {n}")
    print(" sample:", sample[:5], "\n")

In [ ]:
def normalize_docs(rows, repo_name):
    docs = []
    for i, row in enumerate(rows):
        text = (row.get("content") or "").strip()
        if not text:
            continue

        filename = row.get("filename", "")
        docs.append({
            "id": f"{repo_name}-{i}",
            "repo": repo_name,
            "filename": filename,
            "text": text,
        })
    return docs

In [ ]:
import re


def sliding_window_text(text, size=1000, step=800):
    """Return overlapping text windows from a long string."""
    windows = []
    n = len(text)

    for start in range(0, n, step):
        end = min(start + size, n)
        chunk = text[start:end].strip()
        if chunk:
            windows.append({"start": start, "end": end, "text": chunk})
        if end == n:
            break

    return windows


def simple_fixed_chunking(text, chunk_size=1000, overlap=200):
    """Simple character-based chunking with overlap."""
    step = max(chunk_size - overlap, 1)
    windows = sliding_window_text(text, size=chunk_size, step=step)
    return [w["text"] for w in windows]


def paragraph_sliding_chunking(text, paragraph_window=3, paragraph_step=2):
    """Chunk by paragraphs, then use sliding windows across paragraphs."""
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    if not paragraphs:
        return []

    chunks = []
    n = len(paragraphs)
    for i in range(0, n, paragraph_step):
        window = paragraphs[i:i + paragraph_window]
        if not window:
            continue
        chunk = "\n\n".join(window).strip()
        if chunk:
            chunks.append(chunk)
        if i + paragraph_window >= n:
            break

    return chunks


def section_chunking(text):
    """Chunk by markdown headings (#, ##, ###...)."""
    heading_pattern = re.compile(r"^#{1,6}\s+.*$", flags=re.MULTILINE)
    matches = list(heading_pattern.finditer(text))

    if not matches:
        # Fallback when no headings exist.
        return simple_fixed_chunking(text, chunk_size=1200, overlap=200)

    chunks = []
    for idx, match in enumerate(matches):
        start = match.start()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(text)
        section_text = text[start:end].strip()
        if section_text:
            chunks.append(section_text)

    return chunks

In [ ]:
def build_chunks(documents, strategy="simple", **kwargs):
    """Build chunks with one of: simple, paragraph_sliding, section."""
    all_chunks = []

    for doc in documents:
        text = doc["text"]

        if strategy == "simple":
            pieces = simple_fixed_chunking(
                text,
                chunk_size=kwargs.get("chunk_size", 1000),
                overlap=kwargs.get("overlap", 200),
            )
        elif strategy == "paragraph_sliding":
            pieces = paragraph_sliding_chunking(
                text,
                paragraph_window=kwargs.get("paragraph_window", 3),
                paragraph_step=kwargs.get("paragraph_step", 2),
            )
        elif strategy == "section":
            pieces = section_chunking(text)
        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        for idx, piece in enumerate(pieces):
            all_chunks.append(
                {
                    "chunk_id": f'{doc["id"]}-{strategy}-c{idx}',
                    "repo": doc["repo"],
                    "filename": doc["filename"],
                    "chunk_index": idx,
                    "strategy": strategy,
                    "text": piece,
                }
            )

    return all_chunks

In [ ]:
all_docs = []
all_docs += normalize_docs(dtc_langchain, "langchain")
all_docs += normalize_docs(dtc_microsoft, "semantic-kernel")
all_docs += normalize_docs(dtc_openai, "openai-cookbook")

simple_chunks = build_chunks(all_docs, strategy="simple", chunk_size=1000, overlap=200)
paragraph_chunks = build_chunks(
    all_docs,
    strategy="paragraph_sliding",
    paragraph_window=3,
    paragraph_step=2,
)
section_chunks = build_chunks(all_docs, strategy="section")

print("Docs:", len(all_docs))
print("Simple chunks:", len(simple_chunks))
print("Paragraph+sliding chunks:", len(paragraph_chunks))
print("Section chunks:", len(section_chunks))

all_chunks_by_strategy = {
    "simple": simple_chunks,
    "paragraph_sliding": paragraph_chunks,
    "section": section_chunks,
}

for name, chunks in all_chunks_by_strategy.items():
    avg_len = sum(len(c["text"]) for c in chunks) / max(len(chunks), 1)
    print(f"{name} average chunk length: {avg_len:.1f}")

In [ ]:
def preview_chunks(chunks, n=3, max_chars=350):
    for i, ch in enumerate(chunks[:n]):
        print(f"[{i}] {ch['repo']} | {ch['filename']} | len={len(ch['text'])}")
        print(ch["text"][:max_chars].replace("\n", " "))
        print("-" * 80)

print("\n=== SIMPLE SAMPLE ===")
preview_chunks(simple_chunks, n=2)

print("\n=== PARAGRAPH + SLIDING SAMPLE ===")
preview_chunks(paragraph_chunks, n=2)

print("\n=== SECTION SAMPLE ===")
preview_chunks(section_chunks, n=2)

In [ ]:
# Search layer: text search, vector search, and hybrid search
import numpy as np
from sentence_transformers import SentenceTransformer
from minsearch import Index, VectorSearch

# Choose one chunking strategy for retrieval.
search_chunks = section_chunks

# 1) Text search (exact/keyword matching)
text_index = Index(
    text_fields=["text", "filename", "repo"],
    keyword_fields=["repo", "strategy"],
)
text_index.fit(search_chunks)

# 2) Vector search (semantic similarity)
embedding_model = SentenceTransformer("multi-qa-distilbert-cos-v1")

chunk_embeddings = []
for d in search_chunks:
    v = embedding_model.encode(d["text"])
    chunk_embeddings.append(v)

chunk_embeddings = np.array(chunk_embeddings)

vector_index = VectorSearch()
vector_index.fit(chunk_embeddings, search_chunks)

# 3) Hybrid search (combine text + vector)
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)


def vector_search(query, num_results=5):
    q = embedding_model.encode(query)
    return vector_index.search(q, num_results=num_results)


def hybrid_search(query, num_results=5):
    text_results = text_search(query, num_results=num_results)
    vector_results = vector_search(query, num_results=num_results)

    # Weighted Reciprocal Rank Fusion (RRF)
    scores = {}
    items = {}

    for rank, item in enumerate(text_results, start=1):
        key = item["chunk_id"]
        items[key] = item
        scores[key] = scores.get(key, 0.0) + 0.7 * (1.0 / (60 + rank))

    for rank, item in enumerate(vector_results, start=1):
        key = item["chunk_id"]
        items[key] = item
        scores[key] = scores.get(key, 0.0) + 1.0 * (1.0 / (60 + rank))

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [items[key] for key, _ in ranked[:num_results]]


# Quick smoke test
query = "How do I use tools with an agent in LangChain?"
print("TEXT SEARCH")
for r in text_search(query, num_results=3):
    print(f"- {r['repo']} | {r['filename']} | {r['chunk_id']}")

print("\nVECTOR SEARCH")
for r in vector_search(query, num_results=3):
    print(f"- {r['repo']} | {r['filename']} | {r['chunk_id']}")

print("\nHYBRID SEARCH")
for r in hybrid_search(query, num_results=3):
    print(f"- {r['repo']} | {r['filename']} | {r['chunk_id']}")


In [ ]:
# Set your key only in local runtime (do NOT commit real keys)
%env OPENAI_API_KEY=YOUR_OPENAI_API_KEY

In [ ]:
# Day 4: Agent + tools (text/vector/hybrid search)
import json
import os
from openai import OpenAI

openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError(
        "OPENAI_API_KEY is not set for this notebook kernel. "
        "Run `%env OPENAI_API_KEY=...` and re-run this cell."
    )

client = OpenAI(api_key=openai_api_key)


def format_context(results, max_chars=1200):
    parts = []
    for i, r in enumerate(results, start=1):
        text = (r.get("text") or "").strip().replace("\n", " ")
        parts.append(
            f"[{i}] repo={r.get('repo')} file={r.get('filename')} chunk_id={r.get('chunk_id')}\n"
            f"{text[:max_chars]}"
        )
    return "\n\n".join(parts)


def search_tool(query, mode="hybrid", num_results=5):
    if mode == "text":
        return text_search(query, num_results=num_results)
    if mode == "vector":
        return vector_search(query, num_results=num_results)
    return hybrid_search(query, num_results=num_results)


def agent_answer(user_query, mode="hybrid", num_results=5, model="gpt-4o-mini"):
    results = search_tool(user_query, mode=mode, num_results=num_results)
    context = format_context(results)

    system_prompt = (
        "You are a helpful technical assistant. "
        "Answer only from provided context. "
        "If context is insufficient, say what is missing."
    )

    prompt = (
        f"User question:\n{user_query}\n\n"
        f"Retrieved context ({mode} search):\n{context}\n\n"
        "Give a concise answer and cite file paths used."
    )

    response = client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
    )

    return {
        "answer": response.output_text,
        "results": results,
    }


# Quick demo
question = "How can I build an agent that uses tools in these repos?"
out = agent_answer(question, mode="hybrid", num_results=4)

print(out["answer"])
print("\nTop sources:")
for r in out["results"]:
    print(f"- {r['repo']} | {r['filename']} | {r['chunk_id']}")


In [ ]:
# Day 4 (part 1): OpenAI function calling with search tools
import json

search_tool_schema = {
    "type": "function",
    "name": "search_docs",
    "description": "Search project docs with text, vector, or hybrid mode.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "User query to search in docs",
            },
            "mode": {
                "type": "string",
                "enum": ["text", "vector", "hybrid"],
                "description": "Search mode",
            },
            "num_results": {
                "type": "integer",
                "minimum": 1,
                "maximum": 10,
                "description": "How many snippets to fetch",
            },
        },
        "required": ["query"],
        "additionalProperties": False,
    },
}


def run_search_docs(query, mode="hybrid", num_results=5):
    mode = mode or "hybrid"
    num_results = int(num_results or 5)

    if mode == "text":
        results = text_search(query, num_results=num_results)
    elif mode == "vector":
        results = vector_search(query, num_results=num_results)
    else:
        results = hybrid_search(query, num_results=num_results)

    # Keep tool payload compact.
    compact = []
    for r in results:
        compact.append(
            {
                "repo": r.get("repo"),
                "filename": r.get("filename"),
                "chunk_id": r.get("chunk_id"),
                "text": (r.get("text") or "")[:1000],
            }
        )
    return compact


def answer_with_function_calling(user_query, model="gpt-4o-mini"):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a technical assistant. For repo/doc questions, call search_docs first. "
                "Then answer using only returned snippets and cite filenames."
            ),
        },
        {"role": "user", "content": user_query},
    ]

    response = client.responses.create(
        model=model,
        input=messages,
        tools=[search_tool_schema],
    )

    # Handle tool calls (single or multiple).
    for item in response.output:
        if getattr(item, "type", None) == "function_call" and item.name == "search_docs":
            args = json.loads(item.arguments)
            tool_result = run_search_docs(**args)
            messages.append(item)
            messages.append(
                {
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": json.dumps(tool_result),
                }
            )

    final = client.responses.create(
        model=model,
        input=messages,
        tools=[search_tool_schema],
    )
    return final.output_text


# Demo
question = "How do these repos implement tool calling for agents?"
print(answer_with_function_calling(question))


In [ ]:
# Day 4 (part 2): PydanticAI agent with search tool
# If needed once: %pip install pydantic-ai
from pydantic_ai import Agent


def pyd_search(query: str, mode: str = "hybrid", num_results: int = 5) -> list[dict]:
    return run_search_docs(query=query, mode=mode, num_results=num_results)


pyd_agent = Agent(
    "openai:gpt-4o-mini",
    system_prompt=(
        "You are a technical assistant for these repositories. "
        "Use the tool to fetch context before answering repo/document questions. "
        "Cite filenames in your response."
    ),
)


@pyd_agent.tool
async def search_docs(ctx, query: str, mode: str = "hybrid", num_results: int = 5) -> list[dict]:
    return pyd_search(query=query, mode=mode, num_results=num_results)


# Demo (Jupyter-safe: use await, not run_sync)
result = await pyd_agent.run("Where can I find examples of tool-based agents in these repos?")
print(result.output)
